## Before running

In order to keep the GA code consistently usable on Avon, I have created a virtual environment consistent with the python packages on Avon. To use this environment you need to install pipenv with 
- 'pip install pipenv'

Then to create a virtual environment with all the required packages run
- 'pipenv install'

To activate the environment run
- 'pipenv shell'

You can now run the GA code. To exit the virtual environment just use
- 'exit'

All the required packages are listed in 'SOAP_GAS_TMS/Pipfile'

In [ ]:
from genetic_algorithm import *

## Inputs

Dictionaries are taken as input from a parameter file, they contain the parameters for each soap descriptor

In [ ]:
descDict1 = {'lower': 1, 'upper': 10, 'centres': '{6}',
             'neighbours': '{6}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50, 
             'min_cutoff': 4, 'max_cutoff': 10, 'min_sigma': 0.1, 
             'max_sigma': 0.9, 'message_steps': 0}

descDict2 = {'lower': 1, 'upper': 5, 'centres': '{6}',
             'neighbours': '{6}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50, 
             'min_cutoff': 4, 'max_cutoff': 5, 'min_sigma': 0.1, 
             'max_sigma': 0.9, 'message_steps': 0}

Other parameters are also taken as input. These are automatically checked that the parameters are viable

In [ ]:
num_gens = 5
best_sample, lucky_few, population_size, number_of_children = 4, 2, 12, 4
early_stop = 2
early_number = 3 
min_generations = 5

## GeneParameter

GeneParameter class is created from each descriptor dictionary. 

In [ ]:
params1 = GeneParameters(**descDict1)
params2 = GeneParameters(**descDict2)

In [ ]:
params1

## GeneSet

We can use these classes to create a specific set of parameters that are consistant with these values. This returns a randomly generated GeneSet class

In [ ]:
example_gene_set = params1.make_gene_set()
example_gene_set

We can get the parameters used to create the GeneSet class

In [ ]:
example_gene_set.gene_parameters

We can get a descriptor string to be used as an input for getting SOAPs

In [ ]:
example_gene_set.get_soap_string()

We can also mutate the gene using the mutation chance in the GeneParameters class

In [ ]:
print(f"Before mutation {example_gene_set}")
example_gene_set.mutate_gene()
print(f"After mutation {example_gene_set}")

## Individual

An Individual is made up of a list of GeneSet classes.

In [ ]:
example_gene_set_two = params2.make_gene_set()
gene_set_list = [example_gene_set, example_gene_set_two]
example_individual = Individual(gene_set_list[:1])
example_individual

Getting the score for an indivudual

In [ ]:
example_individual.get_score()
example_individual.score

Breeding two individuals to create a child. Mutation is automatically performed during this

In [ ]:
example_individual_two = Individual(gene_set_list)
print(f"Breeding {example_individual} with {example_individual_two}")
child = breed_individuals(example_individual, example_individual_two)
print(f"Created child {child}")

## Population

A Population is a collection of Individual classes. This can be created using a list of GeneParameter classes

In [ ]:
gene_parameters = [params1, params2]
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)
pop

To initialise the population

In [ ]:
pop.initialise_population()

If you want a way of neatly seeing what individuals are in the population

In [ ]:
pop.print_population()

The next generation can then be generated 

In [ ]:
pop.next_generation()
pop.print_population()

So to run the full GA 

In [ ]:
for _ in range(num_gens):
    pop.next_generation()
pop.print_population()

## BestHistory

BestHistory is a class to store the history and check convergence criteria. So the entire GA can be run, printed, and saved using the following code snippet:

In [ ]:
hist = BestHistory(early_stop, early_number, min_generations)
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)

for gen in range(num_gens):
    if hist.converged:
        break
    print(f"Generation {gen}")
    pop.next_generation()
    hist.append(pop)
    write_to_outfile("-------")

There now exists the entire history of the best Individuals throughout each generation that can be saved and easily accessed. 

In [ ]:
vars(hist)

### Running the GA as a script

To run the GA as a script, you just need an input file. See 'SOAP_GAS_TMS/input.py' for an example of the format. 

Then to run the GA with this input file, run 'python genetic_algorithm.py \<input file name without .py extension>'

Running the script creates two files, out_\<input file name>.txt and history_\<input file name>.pkl

If these files already exist, the old ones will be backed up and new ones created. 

The out file contains a log of everything happening in the GA. 
The history file contains a BestHistory object containing the best individuals throughout the GA.

### Testing

In [ ]:
from genetic_algorithm import *
input_parameters = __import__('input')

In [ ]:
gene_parameters = [GeneParameters(**params) for params in input_parameters.descList]

In [ ]:
example_gene_set = [params.make_gene_set() for params in gene_parameters]

In [ ]:
example_gene_set[0].cutoff = 5

In [ ]:
example_individual = Individual(example_gene_set)

In [6]:
# example_individual.cutoff = 5
s = 'soap average cutoff=3 l_max=2 n_max=2 atom_sigma=0.52 n_Z=7 Z={8, 7, 6, 1, 16, 17, 9} n_species=7 species_Z={8, 7, 6, 1, 16, 17, 9} nu_R=1 nu_S=1'
example_individual

Individual(['GeneSet(5, 6, 8, 0.11)'])

In [7]:
example_individual.soap_string_list = [s]

In [8]:
vars(example_individual)

{'gene_set_list': [GeneSet(5, 6, 8, 0.11)],
 'score': 0,
 'soap_string_list': ['soap average cutoff=3 l_max=2 n_max=2 atom_sigma=0.52 n_Z=7 Z={8, 7, 6, 1, 16, 17, 9} n_species=7 species_Z={8, 7, 6, 1, 16, 17, 9} nu_R=1 nu_S=1'],
 'soaps': None,
 'targets': None,
 'regression': None,
 'results_dictionary': defaultdict(list, {})}

In [9]:
conf_s, data = get_conf()
example_individual.comp_soaps(conf_s, data)

In [10]:
example_individual.get_score()

Epoch 1/200
4/4 [==============================] - 0s 43ms/step - loss: 1.1230 - accuracy: 0.2451 - val_loss: 1.0523 - val_accuracy: 0.6154
Epoch 2/200
4/4 [==============================] - 0s 6ms/step - loss: 1.0640 - accuracy: 0.4412 - val_loss: 1.0096 - val_accuracy: 0.5000
Epoch 3/200
4/4 [==============================] - 0s 6ms/step - loss: 1.0389 - accuracy: 0.4412 - val_loss: 0.9795 - val_accuracy: 0.5000
Epoch 4/200
4/4 [==============================] - 0s 6ms/step - loss: 1.0165 - accuracy: 0.4608 - val_loss: 0.9517 - val_accuracy: 0.5385
Epoch 5/200
4/4 [==============================] - 0s 6ms/step - loss: 0.9943 - accuracy: 0.5392 - val_loss: 0.9307 - val_accuracy: 0.6538
Epoch 6/200
4/4 [==============================] - 0s 6ms/step - loss: 0.9740 - accuracy: 0.6176 - val_loss: 0.9057 - val_accuracy: 0.7308
Epoch 7/200
4/4 [==============================] - 0s 6ms/step - loss: 0.9496 - accuracy: 0.5980 - val_loss: 0.8813 - val_accuracy: 0.6923
Epoch 8/200
4/4 [=========

4/4 [==============================] - 0s 5ms/step - loss: 0.5494 - accuracy: 0.7549 - val_loss: 0.5584 - val_accuracy: 0.7692
Epoch 60/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5428 - accuracy: 0.7745 - val_loss: 0.5521 - val_accuracy: 0.8077
Epoch 61/200
4/4 [==============================] - 0s 7ms/step - loss: 0.5187 - accuracy: 0.7647 - val_loss: 0.5774 - val_accuracy: 0.7308
Epoch 62/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5256 - accuracy: 0.7843 - val_loss: 0.5857 - val_accuracy: 0.7692
Epoch 63/200
4/4 [==============================] - 0s 7ms/step - loss: 0.5117 - accuracy: 0.7647 - val_loss: 0.5718 - val_accuracy: 0.7692
Epoch 64/200
4/4 [==============================] - 0s 7ms/step - loss: 0.5034 - accuracy: 0.7941 - val_loss: 0.5298 - val_accuracy: 0.8077
Epoch 65/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5064 - accuracy: 0.7843 - val_loss: 0.5234 - val_accuracy: 0.8462
Epoch 66/200
4/4 [===============

4/4 [==============================] - 0s 6ms/step - loss: 0.2546 - accuracy: 0.8922 - val_loss: 0.7309 - val_accuracy: 0.7308
Epoch 118/200
4/4 [==============================] - 0s 6ms/step - loss: 0.2373 - accuracy: 0.9118 - val_loss: 0.7530 - val_accuracy: 0.7308
Epoch 119/200
4/4 [==============================] - 0s 7ms/step - loss: 0.2307 - accuracy: 0.9118 - val_loss: 0.7423 - val_accuracy: 0.7308
Epoch 120/200
4/4 [==============================] - 0s 7ms/step - loss: 0.2286 - accuracy: 0.9314 - val_loss: 0.7515 - val_accuracy: 0.7308
Epoch 121/200
4/4 [==============================] - 0s 7ms/step - loss: 0.2236 - accuracy: 0.9314 - val_loss: 0.8020 - val_accuracy: 0.7308
Epoch 122/200
4/4 [==============================] - 0s 7ms/step - loss: 0.2258 - accuracy: 0.9118 - val_loss: 0.8292 - val_accuracy: 0.7308
Epoch 123/200
4/4 [==============================] - 0s 6ms/step - loss: 0.2263 - accuracy: 0.9118 - val_loss: 0.7675 - val_accuracy: 0.7308
Epoch 124/200
4/4 [========

4/4 [==============================] - 0s 6ms/step - loss: 0.5058 - accuracy: 0.8039 - val_loss: 1.0915 - val_accuracy: 0.5385
Epoch 44/200
4/4 [==============================] - 0s 6ms/step - loss: 0.4984 - accuracy: 0.8137 - val_loss: 1.2206 - val_accuracy: 0.6154
Epoch 45/200
4/4 [==============================] - 0s 6ms/step - loss: 0.4821 - accuracy: 0.8039 - val_loss: 1.3301 - val_accuracy: 0.5769
Epoch 46/200
4/4 [==============================] - 0s 7ms/step - loss: 0.4869 - accuracy: 0.8039 - val_loss: 1.3145 - val_accuracy: 0.5769
Epoch 47/200
4/4 [==============================] - 0s 6ms/step - loss: 0.4660 - accuracy: 0.8431 - val_loss: 1.2272 - val_accuracy: 0.6154
Epoch 48/200
4/4 [==============================] - 0s 6ms/step - loss: 0.4725 - accuracy: 0.7843 - val_loss: 1.2485 - val_accuracy: 0.6154
Epoch 49/200
4/4 [==============================] - 0s 6ms/step - loss: 0.4622 - accuracy: 0.8137 - val_loss: 1.2116 - val_accuracy: 0.5769
Epoch 50/200
4/4 [===============

4/4 [==============================] - 0s 6ms/step - loss: 0.6415 - accuracy: 0.7255 - val_loss: 1.0666 - val_accuracy: 0.6154
Epoch 35/200
4/4 [==============================] - 0s 6ms/step - loss: 0.6372 - accuracy: 0.7255 - val_loss: 0.9404 - val_accuracy: 0.5769
Epoch 36/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5994 - accuracy: 0.7549 - val_loss: 0.9473 - val_accuracy: 0.5769
Epoch 37/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5953 - accuracy: 0.7843 - val_loss: 0.9889 - val_accuracy: 0.5385
Epoch 38/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5850 - accuracy: 0.7941 - val_loss: 0.9646 - val_accuracy: 0.5385
Epoch 39/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5862 - accuracy: 0.7745 - val_loss: 0.9555 - val_accuracy: 0.5385
Epoch 40/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5885 - accuracy: 0.7843 - val_loss: 1.0108 - val_accuracy: 0.5769
Epoch 41/200
4/4 [===============

4/4 [==============================] - 0s 6ms/step - loss: 0.2689 - accuracy: 0.9020 - val_loss: 1.3238 - val_accuracy: 0.6538
Epoch 93/200
4/4 [==============================] - 0s 843us/step - loss: 0.2540 - accuracy: 0.9412
Epoch 1/200
4/4 [==============================] - 0s 34ms/step - loss: 1.1066 - accuracy: 0.3301 - val_loss: 1.0710 - val_accuracy: 0.2400
Epoch 2/200
4/4 [==============================] - 0s 5ms/step - loss: 1.0538 - accuracy: 0.4854 - val_loss: 1.0143 - val_accuracy: 0.5200
Epoch 3/200
4/4 [==============================] - 0s 6ms/step - loss: 1.0258 - accuracy: 0.5534 - val_loss: 0.9719 - val_accuracy: 0.5200
Epoch 4/200
4/4 [==============================] - 0s 6ms/step - loss: 0.9971 - accuracy: 0.5825 - val_loss: 0.9689 - val_accuracy: 0.5600
Epoch 5/200
4/4 [==============================] - 0s 6ms/step - loss: 0.9706 - accuracy: 0.5631 - val_loss: 0.9508 - val_accuracy: 0.5200
Epoch 6/200
4/4 [==============================] - 0s 6ms/step - loss: 0.9426

4/4 [==============================] - 0s 6ms/step - loss: 0.5208 - accuracy: 0.8155 - val_loss: 0.9206 - val_accuracy: 0.5600
Epoch 57/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5332 - accuracy: 0.7961 - val_loss: 0.8168 - val_accuracy: 0.6000
Epoch 58/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5208 - accuracy: 0.8155 - val_loss: 0.9070 - val_accuracy: 0.6400
Epoch 59/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5017 - accuracy: 0.8155 - val_loss: 0.9445 - val_accuracy: 0.5600
Epoch 60/200
4/4 [==============================] - 0s 6ms/step - loss: 0.4811 - accuracy: 0.8350 - val_loss: 0.8920 - val_accuracy: 0.6000
Epoch 61/200
4/4 [==============================] - 0s 6ms/step - loss: 0.4876 - accuracy: 0.8350 - val_loss: 0.8739 - val_accuracy: 0.6000
Epoch 62/200
4/4 [==============================] - 0s 6ms/step - loss: 0.4809 - accuracy: 0.8350 - val_loss: 0.9596 - val_accuracy: 0.6400
Epoch 63/200
4/4 [===============

4/4 [==============================] - 0s 6ms/step - loss: 0.5774 - accuracy: 0.7379 - val_loss: 0.9424 - val_accuracy: 0.6400
Epoch 41/200
4/4 [==============================] - 0s 6ms/step - loss: 0.5813 - accuracy: 0.7573 - val_loss: 0.9667 - val_accuracy: 0.5600
Epoch 42/200
4/4 [==============================] - 0s 7ms/step - loss: 0.5662 - accuracy: 0.7670 - val_loss: 0.9843 - val_accuracy: 0.5200
Epoch 43/200
4/4 [==============================] - 0s 7ms/step - loss: 0.5563 - accuracy: 0.7670 - val_loss: 0.9558 - val_accuracy: 0.6000
Epoch 44/200
4/4 [==============================] - 0s 7ms/step - loss: 0.5608 - accuracy: 0.7573 - val_loss: 0.9706 - val_accuracy: 0.6000
Epoch 45/200
4/4 [==============================] - 0s 7ms/step - loss: 0.5547 - accuracy: 0.7573 - val_loss: 0.9820 - val_accuracy: 0.5200
Epoch 46/200
4/4 [==============================] - 0s 7ms/step - loss: 0.5628 - accuracy: 0.7767 - val_loss: 0.9744 - val_accuracy: 0.5200
Epoch 47/200
4/4 [===============

In [12]:
example_individual.score

-1.049116063117981

In [ ]:
# example_individual.soap_string_list = ['soap average cutoff=5 l_max=6 n_max=6 atom_sigma=0.22 n_Z=1 Z={6} n_species=1 species_Z={6} mu=0 mu_hat=1 nu=1 nu_hat=0'

In [ ]:
# for mol in conf_s[:5]:
#         soap = []
#         a_soap = descriptors.Descriptor(s).calc(mol)['data']

In [ ]:
mol = conf_s[1]

In [ ]:
a = descriptors.Descriptor(s).calc(mol)['data']

In [ ]:
a.shape

In [ ]:
conf_s, data = get_conf()

In [ ]:
column_names = data.columns
column_names

In [ ]:
try:
    if column_names[1].upper().strip() == 'TARGET':
        regression = True
    elif column_names[1].upper().strip() == 'CLASS':
        regression = False
except:
    raise TypeError("Your database.csv file isc not of the correct "
                     "format. Please read the docs.")

In [ ]:
from genetic_algorithm import *
input_parameters = __import__('input')

conf_s, data = get_conf()
soap_array = []
targets = data.iloc[:, 1].values
s = 'soap average cutoff=3 l_max=2 n_max=2 atom_sigma=0.52 n_Z=7 Z={8, 7, 6, 1, 16, 17, 9} n_species=7 species_Z={8, 7, 6, 1, 16, 17, 9} nu_R=1 nu_S=1'
soap_string_list = [s]
soap_string_list

In [ ]:


soap_array = []
for mol in conf_s:
    soap = []
    for parameter_string in soap_string_list:
        soap += list(descriptors.Descriptor(parameter_string).calc(mol)['data'][0])
        soap_array.append(soap)
    

In [ ]:
len(soap_array)

In [ ]:
vars(mol)

In [ ]:
conf_s[45:46]

In [ ]:
data[45:46]

In [ ]:
soaps = example_individual.soaps
cv = RepeatedKFold(n_splits=5, n_repeats=1, random_state=35)
for train_index, test_index in cv.split(soaps):
    estimator = build_model(soaps, regression=False)
    Y = example_individual.targets
    encoder = LabelEncoder()
    encoder.fit(Y)
    encoded_Y = encoder.transform(Y)
    y = to_categorical(encoded_Y)

    X_train, X_test, X_scaler = scale_data(soaps[train_index],
                                           soaps[test_index])
    y_train, y_test = y[train_index], y[test_index]
    # res = scorer_NN_class(estimator, X_train, X_test, y_train, y_test)

In [ ]:
callback = EarlyStopping(monitor='val_loss', patience=50)
# y_train = np.argmax(y_train, axis=1)
# y_test = np.argmax(y_test, axis=1)
estimator.fit(X_train, y_train, callbacks=[callback], validation_data=(
    X_test, y_test), epochs=200, verbose=True)
y_test_pred = estimator.predict(X_test)
y_train_pred = estimator.predict(X_train)
_, test_accuracy = estimator.evaluate(X_test, y_test_pred)
_, train_accuracy = estimator.evaluate(X_train, y_train_pred)
y_test_pred = np.argmax(y_test_pred, axis=1)
y_train_pred = np.argmax(y_train_pred, axis=1)
score = -1 * (test_accuracy + (0.5 * train_accuracy))
res = [('scores', score), ('y_train_actual', y_train), \
      ('y_test_actual', y_test), ('y_train_predictions', y_train_pred),
           ('y_test_predictions', y_test_pred)]

In [ ]:
res

In [ ]:
y_train.shape

In [ ]:
# estimator.fit(X_train, y[, callbacks=[callback], validation_data=(
#     X_test, y_test), epochs=200, verbose=True)